# NTU Mutual 11 (X-View): probe v1 + v2 on the released CHASE checkpoint
Inference only. Needs the **raw NTU RGB+D 60 skeletons** (`nturgbd_skeletons_s001_to_s017.zip`, from the ROSE Lab request page, https://rose1.ntu.edu.sg/dataset/actionRecognition) uploaded as a Kaggle dataset and attached with **Add Input**. Only the 11 mutual classes (A050–A060) are extracted and preprocessed.

**Settings:** GPU T4, Internet On. Then **Save Version → Save & Run All**. About 30–60 min.

**Send back:** `probe_ntu11/summary.json` and `probe_ntu11_v2/summary_v2.json`.

## 1. Setup

In [ ]:
import os, sys, re, glob, json, time, zipfile, tarfile, pickle, shutil, subprocess
def sh(cmd, cwd=None):
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

WORK = '/kaggle/working'
REPO = WORK + '/CHASE/NeurIPS24'
if not os.path.isdir(REPO):
    sh('git clone -q --depth 1 https://github.com/Necolizer/CHASE.git', cwd=WORK)
sh('pip install -q dotmap einops pyyaml')
import torch, yaml
print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Turn on GPU: Session options -> Accelerator -> GPU T4'


## 2. Mutual-class skeletons + statistics filter

In [ ]:
# Raw NTU RGB+D 60 skeletons -> only the 11 mutual classes (A050-A060) are kept.
# Attach via Add Input either the folder of *.skeleton files or nturgbd_skeletons_s001_to_s017.zip
NTU_DIR = REPO + '/data/ntu'
RAW = REPO + '/data/nturgbd_raw/nturgb+d_skeletons'
os.makedirs(RAW, exist_ok=True)
keep = lambda name: 50 <= int(re.search(r'A(\d{3})', name).group(1)) <= 60

if not glob.glob(RAW + '/*.skeleton'):
    files = glob.glob('/kaggle/input/**/*.skeleton', recursive=True)
    if files:
        for f in files:
            if keep(os.path.basename(f)):
                dst = os.path.join(RAW, os.path.basename(f))
                if not os.path.exists(dst): os.symlink(f, dst)
    else:
        zips = [p for p in glob.glob('/kaggle/input/**/*.zip', recursive=True) if 'skeleton' in p.lower()]
        assert zips, 'Attach the NTU-60 skeleton files or nturgbd_skeletons_s001_to_s017.zip (Add Input)'
        with zipfile.ZipFile(zips[0]) as z:
            for m in z.namelist():
                if m.endswith('.skeleton') and keep(os.path.basename(m)):
                    with z.open(m) as fi, open(os.path.join(RAW, os.path.basename(m)), 'wb') as fo:
                        shutil.copyfileobj(fi, fo)
print('skeleton files (A050-A060):', len(glob.glob(RAW + '/*.skeleton')))

# filter the row-aligned statistics files to the same samples
st = NTU_DIR + '/statistics'
if not os.path.exists(st + '/_orig_names.txt'):
    shutil.copy(st + '/skes_available_name.txt', st + '/_orig_names.txt')
names = open(st + '/_orig_names.txt').read().split()
idx = [i for i, n in enumerate(names) if keep(n)]
for fn in ('skes_available_name', 'label', 'camera', 'performer', 'replication', 'setup'):
    src = st + f'/_orig_{fn}.txt'
    if not os.path.exists(src):
        shutil.copy(st + f'/{fn}.txt', src)
    rows = open(src).read().split()
    assert len(rows) == len(names), fn
    open(st + f'/{fn}.txt', 'w').write('\n'.join(rows[i] for i in idx) + '\n')
present = {os.path.basename(f)[:-9] for f in glob.glob(RAW + '/*.skeleton')}
missing = [names[i] for i in idx if names[i] not in present]
assert not missing, f'{len(missing)} mutual-class skeleton files missing, e.g. {missing[:3]}'
print('samples kept:', len(idx))

# numpy >= 1.24 removed np.int / np.float / np.bool / np.object / np.string_
for f in glob.glob(NTU_DIR + '/*.py'):
    s = open(f).read()
    s2 = re.sub(r'np\.(int|float|bool|object)\b(?!_)', r'\1', s).replace('np.string_', 'np.bytes_')
    if s2 != s: open(f, 'w').write(s2)


## 3. Preprocess (CTR-GCN pipeline) → NTU60_CV.npz

In [ ]:
# Preprocess (CTR-GCN pipeline) -> NTU60_CV.npz   (~10-30 min on Kaggle CPU for the 11 classes)
if not os.path.exists(NTU_DIR + '/NTU60_XView.npz') and not os.path.exists(NTU_DIR + '/NTU60_CV.npz'):
    for s in ('get_raw_skes_data.py', 'get_raw_denoised_data.py', 'seq_transformation.py'):
        t0 = time.time(); sh('python ' + s, cwd=NTU_DIR); print(s, f'{(time.time()-t0)/60:.1f} min')
    shutil.rmtree(NTU_DIR + '/raw_data', ignore_errors=True)       # free disk
    shutil.rmtree(NTU_DIR + '/denoised_data', ignore_errors=True)

# The script writes NTU60_XView.npz / NTU60_XSub.npz; the CHASE configs expect NTU60_CV.npz / NTU60_CS.npz
for src, dst in (('NTU60_XView.npz', 'NTU60_CV.npz'), ('NTU60_XSub.npz', 'NTU60_CS.npz')):
    if os.path.exists(f'{NTU_DIR}/{src}') and not os.path.exists(f'{NTU_DIR}/{dst}'):
        os.rename(f'{NTU_DIR}/{src}', f'{NTU_DIR}/{dst}')
print(sorted(f for f in os.listdir(NTU_DIR) if f.endswith('.npz')))

import numpy as np
d = np.load(NTU_DIR + '/NTU60_CV.npz', mmap_mode='r')
print({k: d[k].shape for k in d.files})


## 4. Checkpoint, config, probe scripts

In [ ]:
if not os.path.exists(REPO + '/ntu11_xview_chas.pt'):
    sh('wget -q -O ntu11_xview_chas.pt https://huggingface.co/Necolizer/CHASE/resolve/main/NTU11_XView_CTR-GCN_CHASE/ntu11_xview_chas.pt', cwd=REPO)
cfg = yaml.safe_load(open(REPO + '/config/CTRGCN/ntu11/ntu11_xview_joint_chase.yaml'))
cfg['test_feeder_args']['data_path'] = NTU_DIR + '/NTU60_CV.npz'
cfg['train_feeder_args']['data_path'] = NTU_DIR + '/NTU60_CV.npz'
cfg['test_batch_size'] = 64
yaml.safe_dump(cfg, open(REPO + '/ntu11_probe.yaml', 'w'))
P1 = r'''"""
probe_chase.py -- the two cheap Paper C experiments, run on the RELEASED CHASE
checkpoints. No training, inference only.

  Exp 1  Diagnostic of the hull weights alpha (RBR decision rule, issues a/b, G3)
         N_eff = 1/sum(alpha^2), max alpha, alpha mass on zero-padded points,
         distance of p* from the S2CoM centroid, between-sample spread of p*.
  Exp 2  Exactness tests (Props. 10 and 12)
         (a) test-time global translation of the valid joints: equivariance error
             ||p*(x+t) - p*(x) - t|| / ||t||, accuracy and prediction flips
         (b) test-time permutation of the entity axis: ||p*(x_pi) - p*(x)||,
             accuracy and prediction flips
         CHASE+ (chase_plus.py) gives exactly zero for (a) and, if hierarchical,
         for (b); this probe measures how far the released CHASE is from that.

Run from inside the CHASE repo folder NeurIPS24/ :

  python probe_chase.py --config config/CTRGCN/cad/cad_chase.yaml \
                        --weights cad_chas.pt --out probe_cad
  python probe_chase.py --config config/CTRGCN/ntu11/ntu11_xview_joint_chase.yaml \
                        --weights ntu11_xview_chas.pt --out probe_ntu11

Outputs in --out: summary.json, per_sample.csv, neff_hist.png, translation.csv,
permutation.csv.
"""
import argparse
import csv
import importlib
import json
import math
import os
from collections import OrderedDict

import numpy as np
import torch
import yaml
from torch.utils.data import DataLoader


def import_class(name):
    mod, cls = name.rsplit(".", 1)
    return getattr(importlib.import_module(mod), cls)


def load(cfg_path, weights, device, num_workers=0):
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    model = import_class(cfg["model"])(**cfg["model_args"])
    if cfg.get("wrapper"):
        model = import_class(cfg["wrapper"])(model, **cfg["wrapper_args"])
    else:
        raise SystemExit("config has no CHASE wrapper")
    if list(cfg["wrapper_args"].get("pooling_seg", [1, 1, 1])) != [1, 1, 1]:
        raise SystemExit("probe assumes pooling_seg [1,1,1] (the CHASE default)")
    sd = torch.load(weights, map_location="cpu")
    sd = OrderedDict((k.split("module.")[-1], v) for k, v in sd.items())
    model.load_state_dict(sd, strict=True)
    model.to(device).eval()
    feeder = import_class(cfg["feeder"])(**cfg["test_feeder_args"])
    loader = DataLoader(feeder, batch_size=cfg.get("test_batch_size", 64),
                        shuffle=False, num_workers=num_workers, drop_last=False)
    return cfg, model, loader


@torch.no_grad()
def hull(wrapper, x):
    """Replicates CHASWrapper.forward lines 19-21: alpha (N,U) and p* (N,C)."""
    N, C, T, V, M = x.shape
    alpha = wrapper.shift(x).view(N, T * M * V, -1)[..., 0].softmax(dim=1)
    tx = x.reshape(N, C, T * V * M)                              # (t v m), as rearrange
    pstar = torch.einsum("ncu,nu->nc", tx, alpha)
    return alpha, pstar


def valid_mask(x):
    return x.abs().amax(dim=1) > 0                                  # (N,T,V,M)


def logits_of(model, x):
    out = model(x)
    return out[0] if isinstance(out, tuple) else out


@torch.no_grad()
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--weights", required=True)
    ap.add_argument("--out", default="probe_out")
    ap.add_argument("--sigmas", default="0.1,0.5,1,2,5",
                    help="translation std, in units of the data scale s")
    ap.add_argument("--perm_trials", type=int, default=3)
    ap.add_argument("--max_batches", type=int, default=0, help="0 = whole test set")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--num_workers", type=int, default=0)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    torch.manual_seed(a.seed); np.random.seed(a.seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    cfg, model, loader = load(a.config, a.weights, dev, a.num_workers)
    sigmas = [float(s) for s in a.sigmas.split(",")]

    rows = []
    n_tot = 0
    correct = {"clean": 0}
    tr = {s: dict(correct=0, flips=0, eqerr=[], dvalid=[], dpad=[]) for s in sigmas}
    pe = dict(correct=0, flips=0, n=0, dp=[])

    for b, batch in enumerate(loader):
        if a.max_batches and b >= a.max_batches:
            break
        x = batch[0].float().to(dev)
        y = batch[1].long().to(dev)
        N, C, T, V, M = x.shape
        pm = valid_mask(x)                                          # (N,T,V,M)
        pmf = pm.flatten(1)                                         # (N,U) in (t v m)
        alpha, ps = hull(model, x)
        pred = logits_of(model, x).argmax(1)
        correct["clean"] += (pred == y).sum().item()

        # ---- Exp 1: weight diagnostics
        cnt = pm.flatten(1).sum(1).clamp_min(1)
        cen = (x * pm.unsqueeze(1)).flatten(2).sum(2) / cnt.unsqueeze(1)         # S2CoM (valid pts)
        dev2 = ((x - cen.view(N, C, 1, 1, 1)) ** 2).sum(1)                       # (N,T,V,M)
        scale = torch.sqrt((dev2 * pm).flatten(1).sum(1) / cnt)                  # RMS radius s
        neff = 1.0 / (alpha ** 2).sum(1)
        pad_mass = (alpha * (~pmf)).sum(1)
        for i in range(N):
            rows.append(dict(
                label=int(y[i]), correct=int(pred[i] == y[i]),
                U=int(alpha.shape[1]), U_valid=int(pmf[i].sum()),
                neff=float(neff[i]), max_alpha=float(alpha[i].max()),
                pad_mass=float(pad_mass[i]),
                dist_to_s2com_over_s=float((ps[i] - cen[i]).norm() / scale[i].clamp_min(1e-8)),
                scale=float(scale[i]),
                **{f"pstar_{c}": float(ps[i, c]) for c in range(C)}))

        # ---- Exp 2a: global translation of the valid joints only
        for s in sigmas:
            t = torch.randn(N, C, device=dev) * scale.view(N, 1) * s
            xt = x + t.view(N, C, 1, 1, 1) * pm.unsqueeze(1)
            _, pst = hull(model, xt)
            tn = t.norm(dim=1).clamp_min(1e-8)
            tr[s]["eqerr"] += ((pst - ps - t).norm(dim=1) / tn).tolist()
            # change of the backbone input after the CHASE shift
            xs0 = x - ps.view(N, C, 1, 1, 1); xs1 = xt - pst.view(N, C, 1, 1, 1)
            d = (xs1 - xs0).norm(dim=1) / scale.view(N, 1, 1, 1).clamp_min(1e-8)
            tr[s]["dvalid"] += [float(d[i][pm[i]].mean()) for i in range(N)]
            tr[s]["dpad"] += [float(d[i][~pm[i]].mean()) if (~pm[i]).any() else 0.0 for i in range(N)]
            pt = logits_of(model, xt).argmax(1)
            tr[s]["correct"] += (pt == y).sum().item()
            tr[s]["flips"] += (pt != pred).sum().item()

        # ---- Exp 2b: entity permutation
        if M > 1:
            for _ in range(a.perm_trials):
                perm = torch.arange(M - 1, -1, -1) if M == 2 else torch.randperm(M)
                if M > 2 and torch.equal(perm, torch.arange(M)):
                    perm = torch.roll(perm, 1)
                xp = x[..., perm.to(dev)]
                _, pp = hull(model, xp)
                pe["dp"] += ((pp - ps).norm(dim=1) / scale.clamp_min(1e-8)).tolist()
                pp_pred = logits_of(model, xp).argmax(1)
                pe["correct"] += (pp_pred == y).sum().item()
                pe["flips"] += (pp_pred != pred).sum().item()
                pe["n"] += N
        n_tot += N
        print(f"batch {b}: {n_tot} samples", flush=True)

    # ---------------------------------------------------------------- report
    neff = np.array([r["neff"] for r in rows]); uv = np.array([r["U_valid"] for r in rows])
    U = rows[0]["U"]
    P = np.array([[r[f"pstar_{c}"] for c in range(C)] for r in rows])
    S = np.array([r["scale"] for r in rows])
    q = lambda v: {k: float(np.quantile(v, p)) for k, p in
                   [("p05", .05), ("p25", .25), ("median", .5), ("p75", .75), ("p95", .95)]}
    summary = dict(
        config=a.config, weights=a.weights, n_samples=n_tot, U=U,
        acc_clean=100 * correct["clean"] / n_tot,
        exp1=dict(
            neff=q(neff), neff_over_Uvalid=q(neff / np.maximum(uv, 1)),
            frac_saturated_neff_lt_10=float((neff < 10).mean()),
            frac_overdispersed_neff_gt_half_Uvalid=float((neff > 0.5 * uv).mean()),
            max_alpha=q(np.array([r["max_alpha"] for r in rows])),
            pad_mass=q(np.array([r["pad_mass"] for r in rows])),
            frac_samples_with_padding=float((uv < U).mean()),
            pad_mass_mean_on_padded_samples=float(np.mean([r["pad_mass"] for r in rows if r["U_valid"] < U]))
                if (uv < U).any() else 0.0,
            dist_to_s2com_over_s=q(np.array([r["dist_to_s2com_over_s"] for r in rows])),
            pstar_between_sample_std_over_mean_s=(P.std(0) / S.mean()).tolist()),
        exp2_translation={str(s): dict(
            acc=100 * v["correct"] / n_tot, flip_rate=100 * v["flips"] / n_tot,
            equiv_err_median=float(np.median(v["eqerr"])), equiv_err_p95=float(np.quantile(v["eqerr"], .95)),
            input_change_valid_mean=float(np.mean(v["dvalid"])),
            input_change_padding_mean=float(np.mean(v["dpad"]))) for s, v in tr.items()},
        exp2_permutation=dict(
            acc=100 * pe["correct"] / max(pe["n"], 1), flip_rate=100 * pe["flips"] / max(pe["n"], 1),
            dpstar_over_s=q(np.array(pe["dp"])) if pe["dp"] else None),
        notes=["CHASE+ predicts equiv_err = 0, input_change = 0 and flip_rate = 0 for translation; "
               "hierarchical CHASE+ predicts dpstar = 0 and flip_rate = 0 for permutation.",
               "input_change_padding > 0 even for an exactly equivariant p*: zero padding becomes -p* "
               "in the released wrapper (G3); read equiv_err for the CLB itself."])

    with open(os.path.join(a.out, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)
    with open(os.path.join(a.out, "per_sample.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    with open(os.path.join(a.out, "translation.csv"), "w", newline="") as f:
        w = csv.writer(f); w.writerow(["sigma_over_s", "acc", "flip_rate", "equiv_err_median",
                                       "equiv_err_p95", "input_change_valid", "input_change_padding"])
        for s, v in summary["exp2_translation"].items():
            w.writerow([s, v["acc"], v["flip_rate"], v["equiv_err_median"], v["equiv_err_p95"],
                        v["input_change_valid_mean"], v["input_change_padding_mean"]])
    try:
        import matplotlib; matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
        ax[0].hist(neff, bins=50, color="#4C72B0")
        ax[0].set_xlabel("N_eff = 1 / sum alpha^2"); ax[0].set_ylabel("test samples")
        ax[0].set_title(f"U = {U}, median U_valid = {int(np.median(uv))}")
        ax[1].hist(neff / np.maximum(uv, 1), bins=50, color="#55A868")
        ax[1].set_xlabel("N_eff / U_valid  (1 = uniform = S2CoM)")
        fig.tight_layout(); fig.savefig(os.path.join(a.out, "neff_hist.png"), dpi=150)
    except Exception as e:                                         # plotting is optional
        print("plot skipped:", e)

    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()
'''
P2 = r'''"""
probe_chase_v2.py -- follow-up controls after the first CAD probe.
Needs probe_chase.py in the same folder (reuses its loader).

  A. Which point is the origin?   (N_eff = 1 => alpha is one-hot)
     frame / joint / slot of argmax alpha, whether it is padding, whether it is
     an extreme point (min or max of x or y over the valid points).
  B. Translation control           (is the accuracy drop caused by padding, G3?)
     valid-only  : translate real joints, padding stays 0          (as in v1)
     all-points  : translate real joints AND padding together
     If CHASE's origin is equivariant, all-points must give ~no change.
  C. Permutation decomposition     (CHASE vs backbone order dependence, G4)
     full          : model(x permuted)
     backbone_only : same origin as clean, entities permuted after the shift
     chase_only    : origin from the permuted input, entities kept in clean order
  D. Origin replacement            (how much does the trained model rely on the
     selected point?)  origin := S2CoM centroid of the valid points.

  python probe_chase_v2.py --config cad_probe.yaml --weights cad_chas.pt --out probe_cad_v2
"""
import argparse
import json
import os

import numpy as np
import torch

from probe_chase import hull, load, logits_of, valid_mask


@torch.no_grad()
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--weights", required=True)
    ap.add_argument("--out", default="probe_v2")
    ap.add_argument("--sigmas", default="0.5,1,2")
    ap.add_argument("--perm_trials", type=int, default=3)
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    torch.manual_seed(a.seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    cfg, model, loader = load(a.config, a.weights, dev, 0)
    bb = model.backbone
    sig = [float(s) for s in a.sigmas.split(",")]

    n = 0
    clean_ok = 0
    sel_t, sel_v, sel_m, sel_pad, sel_ext = [], [], [], [], []
    tr = {s: {k: [0, 0] for k in ("valid_only", "all_points")} for s in sig}   # [correct, flips]
    pe = {k: [0, 0] for k in ("full", "backbone_only", "chase_only")}
    pe_n = 0
    rep = [0, 0]

    for batch in loader:
        x = batch[0].float().to(dev); y = batch[1].long().to(dev)
        N, C, T, V, M = x.shape
        keep = valid_mask(x).flatten(1).any(1)                       # drop all-empty samples
        x, y = x[keep], y[keep]
        N = x.shape[0]
        if N == 0:
            continue
        pm = valid_mask(x)
        alpha, ps = hull(model, x)
        pred = logits_of(model, x).argmax(1)
        clean_ok += (pred == y).sum().item()

        # ---- A. selected point
        u = alpha.argmax(1)
        t_i, rem = u // (V * M), u % (V * M)
        v_i, m_i = rem // M, rem % M
        sel_t += t_i.tolist(); sel_v += v_i.tolist(); sel_m += m_i.tolist()
        sel_pad += (~pm.flatten(1).gather(1, u[:, None])[:, 0]).tolist()
        big = torch.finfo(x.dtype).max
        xf = x.flatten(2)                                           # (N,C,U)
        vm = pm.flatten(1).unsqueeze(1)
        mx = xf.masked_fill(~vm, -big).amax(2); mn = xf.masked_fill(~vm, big).amin(2)
        sp = xf.gather(2, u.view(N, 1, 1).expand(N, C, 1))[..., 0]
        sel_ext += ((sp == mx) | (sp == mn)).any(1).tolist()

        # ---- B. translation control
        cnt = pm.flatten(1).sum(1).clamp_min(1)
        cen = (x * pm.unsqueeze(1)).flatten(2).sum(2) / cnt.unsqueeze(1)
        scale = torch.sqrt((((x - cen.view(N, C, 1, 1, 1)) ** 2).sum(1) * pm).flatten(1).sum(1) / cnt)
        for s in sig:
            t = (torch.randn(N, C, device=dev) * scale.view(N, 1) * s).view(N, C, 1, 1, 1)
            for k, xt in (("valid_only", x + t * pm.unsqueeze(1)), ("all_points", x + t)):
                p = logits_of(model, xt).argmax(1)
                tr[s][k][0] += (p == y).sum().item(); tr[s][k][1] += (p != pred).sum().item()

        # ---- C. permutation decomposition
        xs = x - ps.view(N, C, 1, 1, 1)                              # wrapper's input to backbone
        for _ in range(a.perm_trials):
            perm = torch.arange(M - 1, -1, -1) if M == 2 else torch.randperm(M)
            if M > 2 and torch.equal(perm, torch.arange(M)):
                perm = torch.roll(perm, 1)
            perm = perm.to(dev); inv = torch.argsort(perm)
            xp = x[..., perm]
            _, pp = hull(model, xp)
            outs = {
                "full": logits_of(model, xp),
                "backbone_only": logits_of(bb, xs[..., perm]),
                "chase_only": logits_of(bb, (xp - pp.view(N, C, 1, 1, 1))[..., inv]),
            }
            for k, o in outs.items():
                p = o.argmax(1)
                pe[k][0] += (p == y).sum().item(); pe[k][1] += (p != pred).sum().item()
            pe_n += N

        # ---- D. origin := S2CoM
        p = logits_of(bb, x - cen.view(N, C, 1, 1, 1)).argmax(1)
        rep[0] += (p == y).sum().item(); rep[1] += (p != pred).sum().item()
        n += N

    pct = lambda c: round(100 * c / n, 2)
    hist = lambda v, K: np.bincount(np.array(v), minlength=K).tolist()
    s = dict(
        n_samples=n, acc_clean=pct(clean_ok),
        A_selected_point=dict(
            frac_on_padding=round(float(np.mean(sel_pad)), 4),
            frac_extreme_xy=round(float(np.mean(sel_ext)), 4),
            joint_hist=hist(sel_v, V), frame_hist=hist(sel_t, T), slot_hist=hist(sel_m, M)),
        B_translation={str(k): {m: dict(acc=pct(v[0]), flip_rate=pct(v[1])) for m, v in d.items()}
                       for k, d in tr.items()},
        C_permutation={k: dict(acc=round(100 * v[0] / pe_n, 2), flip_rate=round(100 * v[1] / pe_n, 2))
                       for k, v in pe.items()},
        D_origin_s2com=dict(acc=pct(rep[0]), flip_rate=pct(rep[1])))
    json.dump(s, open(os.path.join(a.out, "summary_v2.json"), "w"), indent=2)
    print(json.dumps(s, indent=2))


if __name__ == "__main__":
    main()
'''
open(REPO + '/probe_chase.py', 'w').write(P1)
open(REPO + '/probe_chase_v2.py', 'w').write(P2)
print('ready')


## 5. Run probes

In [ ]:
sh('python probe_chase.py --config ntu11_probe.yaml --weights ntu11_xview_chas.pt --out ' + WORK + '/probe_ntu11', cwd=REPO)
sh('python probe_chase_v2.py --config ntu11_probe.yaml --weights ntu11_xview_chas.pt --out ' + WORK + '/probe_ntu11_v2', cwd=REPO)
print(open(WORK + '/probe_ntu11/summary.json').read())
print(open(WORK + '/probe_ntu11_v2/summary_v2.json').read())
print('Send back: probe_ntu11/summary.json and probe_ntu11_v2/summary_v2.json')
